# RuneScape Bond vs S&P 500 Investigation

This notebook investigates whether Old School RuneScape bond prices contain genuine predictive information about subsequent S&P 500 movement. The standard for evidence is deliberately high: even if a raw correlation exists, the key question is whether it survives basic time-series hygiene, beats what random chance would generate, and remains informative after controlling for more established macro or risk indicators.

The workflow therefore moves from data hygiene to signal discovery to adversarial checks. First, the notebook verifies that the series being compared are statistically suitable for correlation work at all. It then measures lead-lag structure, tests whether the best result is stronger than shuffled data, and asks whether any apparent signal is still present once known indicators are added. The later sections examine predictive directionality, stability across market regimes, and out-of-sample performance so the final conclusion is based on a chain of evidence rather than a single chart.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import ruptures as rpt

from arch.unitroot import ADF, KPSS
from IPython.display import Markdown, display
from pandas_datareader import data as pdr
from statsmodels.tsa.stattools import grangercausalitytests

from bond_sp500_correlation import fetch_rs_timeseries, fetch_sp500

In [ ]:
sns.set_theme(style="whitegrid", palette="crest")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["figure.dpi"] = 120
pd.options.display.float_format = "{:.4f}".format

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

RS_ITEM_ID = 12532
RS_MARKET = "osrs"
RS_TIMESTEP = "24h"
SP500_SYMBOL = "^GSPC"
VIX_SYMBOL = "^VIX"
MAX_LAG_DAYS = 90
SHUFFLE_ITERATIONS = 1000
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## Section 1 — Data Fetching and Caching

Iterative notebook work becomes slow and brittle if every rerun re-hits remote APIs. Caching the raw RuneScape and market datasets locally makes the later sections cheap to rerun, which matters because this notebook repeatedly revisits the same base series while changing statistical tests and visualizations. Caching also protects the investigation from transient API failures and makes the workflow reproducible.

The analysis should start from the earliest RuneScape bond history the API provides rather than from a recent cutoff. A short recent sample risks overfitting to one market regime, while a longer sample covers multiple equity environments including pre-COVID conditions, the COVID crash, the 2022 hiking cycle, and subsequent recovery periods. If any relationship is real, it should be examined across those very different regimes rather than only in the most recent one.

In [ ]:
def _cache_name(prefix: str, *parts: object) -> Path:
    safe = "_".join(str(part).replace("^", "").replace("/", "-") for part in parts)
    return CACHE_DIR / f"{prefix}_{safe}.parquet"


def load_or_fetch_rs_data(
    item_id: int = RS_ITEM_ID,
    market: str = RS_MARKET,
    timestep: str = RS_TIMESTEP,
    refresh: bool = False,
) -> pd.DataFrame:
    cache_path = _cache_name("rs_raw", market, item_id, timestep)
    if cache_path.exists() and not refresh:
        rs_raw = pd.read_parquet(cache_path)
    else:
        rs_raw = fetch_rs_timeseries(item_id=item_id, market=market, timestep=timestep).copy()
        rs_raw.index = pd.to_datetime(rs_raw.index).tz_convert(None)
        rs_raw.to_parquet(cache_path)
    rs_raw = rs_raw.rename(columns={"price": "bond_price"}).sort_index()
    rs_raw.index.name = "date"
    return rs_raw


def load_or_fetch_sp500_data(
    start: pd.Timestamp,
    end: pd.Timestamp,
    symbol: str = SP500_SYMBOL,
    refresh: bool = False,
) -> pd.DataFrame:
    cache_path = _cache_name("market_raw", symbol, start.date(), end.date())
    if cache_path.exists() and not refresh:
        sp500_raw = pd.read_parquet(cache_path)
    else:
        sp500_raw = fetch_sp500(symbol, start=start, end=end).copy()
        sp500_raw.index = pd.to_datetime(sp500_raw.index).tz_localize(None)
        sp500_raw.to_parquet(cache_path)
    sp500_raw = sp500_raw.sort_index()
    sp500_raw.index.name = "date"
    return sp500_raw


def load_or_fetch_fred_series(
    series_code: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    refresh: bool = False,
) -> pd.DataFrame:
    cache_path = _cache_name("fred_raw", series_code, start.date(), end.date())
    if cache_path.exists() and not refresh:
        fred_raw = pd.read_parquet(cache_path)
    else:
        fred_raw = pdr.DataReader(series_code, "fred", start=start, end=end)
        fred_raw.to_parquet(cache_path)
    fred_raw.index = pd.to_datetime(fred_raw.index).tz_localize(None)
    fred_raw.index.name = "date"
    return fred_raw.sort_index()


def load_or_fetch_yfinance_series(
    symbol: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    refresh: bool = False,
) -> pd.DataFrame:
    cache_path = _cache_name("yf_raw", symbol, start.date(), end.date())
    if cache_path.exists() and not refresh:
        market_raw = pd.read_parquet(cache_path)
    else:
        market_raw = fetch_sp500(symbol, start=start, end=end).copy()
        market_raw.to_parquet(cache_path)
    market_raw.index = pd.to_datetime(market_raw.index).tz_localize(None)
    market_raw.index.name = "date"
    if market_raw.columns.tolist() == ["spx_price"]:
        market_raw = market_raw.rename(columns={"spx_price": f"{symbol}_price"})
    return market_raw.sort_index()


def build_analysis_frame(refresh: bool = False) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    bond_raw = load_or_fetch_rs_data(refresh=refresh)
    sp500_raw = load_or_fetch_sp500_data(
        start=bond_raw.index.min(),
        end=bond_raw.index.max(),
        refresh=refresh,
    )
    joint = bond_raw.join(sp500_raw, how="inner")
    joint["bond_return"] = joint["bond_price"].pct_change()
    joint["sp500_return"] = joint["spx_price"].pct_change()
    analysis = joint.dropna(subset=["bond_return", "sp500_return"]).copy()
    return bond_raw, sp500_raw, analysis


def correlation_at_lag(x: pd.Series, y: pd.Series, lag: int) -> float:
    aligned = pd.concat([x.shift(lag).rename("x"), y.rename("y")], axis=1).dropna()
    if len(aligned) < 3:
        return np.nan
    return stats.pearsonr(aligned["x"], aligned["y"])[0]


bond_raw, sp500_raw, analysis_df = build_analysis_frame(refresh=False)

display(bond_raw.head())
display(sp500_raw.head())
display(analysis_df[["bond_price", "spx_price", "bond_return", "sp500_return"]].head())

print(f"Bond history: {bond_raw.index.min().date()} to {bond_raw.index.max().date()} ({len(bond_raw):,} rows)")
print(f"S&P 500 history: {sp500_raw.index.min().date()} to {sp500_raw.index.max().date()} ({len(sp500_raw):,} rows)")
print(f"Shared trading days with returns: {len(analysis_df):,}")

## Section 2 — Stationarity Tests (ADF and KPSS)

Correlation analysis is unreliable on trending, non-stationary price series because two independent series can look related simply because both drift over time. Stationarity means the distributional properties of a series, such as its mean and variance, are stable enough for time-series inference to make sense. That is why the first serious checkpoint is not a correlation chart but a stationarity check.

The expected result is straightforward. Raw asset prices usually look non-stationary because they embed long-run drift, regime changes, and level shifts. Daily returns are usually much closer to stationary, which is why finance work is typically done on returns rather than raw prices. If the tests disagree with that expectation, the notebook should treat it as a warning rather than force the later analysis forward without comment.

In [ ]:
def run_stationarity_suite(series: pd.Series, label: str) -> list[dict[str, object]]:
    clean = series.dropna()
    adf_result = ADF(clean)
    kpss_result = KPSS(clean)
    return [
        {
            "series": label,
            "test": "ADF",
            "statistic": float(adf_result.stat),
            "pvalue": float(adf_result.pvalue),
            "interpretation": "Stationary (reject unit root)" if adf_result.pvalue < 0.05 else "Non-stationary not rejected",
        },
        {
            "series": label,
            "test": "KPSS",
            "statistic": float(kpss_result.stat),
            "pvalue": float(kpss_result.pvalue),
            "interpretation": "Non-stationary (reject stationarity)" if kpss_result.pvalue < 0.05 else "Stationarity not rejected",
        },
    ]


stationarity_records: list[dict[str, object]] = []
stationarity_records.extend(run_stationarity_suite(analysis_df["bond_price"], "Bond price"))
stationarity_records.extend(run_stationarity_suite(analysis_df["spx_price"], "S&P 500 price"))
stationarity_records.extend(run_stationarity_suite(analysis_df["bond_return"], "Bond return"))
stationarity_records.extend(run_stationarity_suite(analysis_df["sp500_return"], "S&P 500 return"))

stationarity_results = pd.DataFrame(stationarity_records)
display(stationarity_results)

for row in stationarity_results.itertuples(index=False):
    print(
        f"{row.series:16s} | {row.test:4s} | stat={row.statistic:9.4f} | "
        f"p-value={row.pvalue:8.4f} | {row.interpretation}"
    )

price_stationary = bool(
    (
        stationarity_results["series"].str.contains("price")
        & stationarity_results["interpretation"].str.contains("Stationary", case=False)
    ).any()
)
return_stationary = bool(
    (
        stationarity_results["series"].str.contains("return")
        & stationarity_results["interpretation"].str.contains("Stationary", case=False)
    ).any()
)

stationary_analysis_cols = ["bond_return", "sp500_return"]

notes = []
if price_stationary:
    notes.append(
        "At least one price-series test came back more stationary than expected, so later interpretation should stay cautious about structural breaks or finite-sample artifacts."
    )
else:
    notes.append(
        "The price-series results behave as expected for financial levels: they should not be treated as the basis for correlation inference."
    )
if return_stationary:
    notes.append(
        "The return-series results are consistent with using daily returns as the working series for the rest of the notebook."
    )
else:
    notes.append(
        "The return-series results are weaker than expected, which raises the bar for every later claim and should be treated as a substantive warning."
    )

display(
    Markdown(
        "## Section 2 Observations\n\n"
        "Going forward, the notebook uses **daily percent returns** as the analysis series because that is the representation most likely to satisfy the stationarity assumptions behind the later tests. "
        + " ".join(notes)
    )
)

## Section 3 — Cross-Correlation and Lag Structure

Before asking whether the bond series is useful, the notebook first asks a narrower question: if there is any relationship at all, where does it sit in time? A signal that only appears when the bond series is shifted relative to the S&P 500 is a very different object from a same-day correlation. Mapping the full lag structure is therefore the foundation for everything that follows.

The shuffle test is an adversarial check on that lag search. When many lags are tried, some lag will usually look best even if there is no real signal. By shuffling one return series many times and repeating the same lag search, the notebook estimates how impressive the best observed lag actually is relative to chance. A lag that does not beat this baseline should be treated as data mining noise rather than evidence.

In [ ]:
lags = np.arange(-MAX_LAG_DAYS, MAX_LAG_DAYS + 1)
correlation_rows = []
for lag in lags:
    shifted = analysis_df["bond_return"].shift(lag)
    aligned = pd.concat([shifted.rename("bond_shifted"), analysis_df["sp500_return"]], axis=1).dropna()
    if len(aligned) < 3:
        continue
    corr, pvalue = stats.pearsonr(aligned["bond_shifted"], aligned["sp500_return"])
    correlation_rows.append({"lag": lag, "correlation": corr, "pvalue": pvalue, "n_obs": len(aligned)})

lag_corr_df = pd.DataFrame(correlation_rows)
best_lag_idx = lag_corr_df["correlation"].abs().idxmax()
best_lag = int(lag_corr_df.loc[best_lag_idx, "lag"])
best_corr = float(lag_corr_df.loc[best_lag_idx, "correlation"])
best_pvalue = float(lag_corr_df.loc[best_lag_idx, "pvalue"])

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(lag_corr_df["lag"], lag_corr_df["correlation"], color=sns.color_palette("crest", 5)[2], width=0.9)
ax.axvline(best_lag, color="#c44e52", linestyle="--", linewidth=2, label=f"Best lag = {best_lag}")
ax.set_title("Cross-correlation of bond returns and S&P 500 returns")
ax.set_xlabel("Lag (days, positive means bond leads S&P 500)")
ax.set_ylabel("Pearson correlation")
ax.legend()
plt.show()

rng = np.random.default_rng(RANDOM_SEED)
shuffled_best_corrs = []
for _ in range(SHUFFLE_ITERATIONS):
    shuffled = pd.Series(rng.permutation(analysis_df["bond_return"].to_numpy()), index=analysis_df.index)
    permuted_corrs = [abs(correlation_at_lag(shuffled, analysis_df["sp500_return"], lag)) for lag in lags]
    shuffled_best_corrs.append(np.nanmax(permuted_corrs))

shuffle_results = pd.Series(shuffled_best_corrs, name="best_abs_corr")
actual_abs_corr = abs(best_corr)
shuffle_pvalue = float((np.sum(shuffle_results >= actual_abs_corr) + 1) / (len(shuffle_results) + 1))

fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(shuffle_results, bins=30, color=sns.color_palette("crest", 5)[1], ax=ax)
ax.axvline(actual_abs_corr, color="#c44e52", linestyle="--", linewidth=2, label=f"Actual |corr| = {actual_abs_corr:.4f}")
ax.set_title("Shuffle-test null distribution of best absolute lag correlation")
ax.set_xlabel("Best absolute correlation after shuffling bond returns")
ax.set_ylabel("Count")
ax.legend()
plt.show()

lag_direction_text = (
    f"RuneScape bonds lead the S&P 500 by {best_lag} trading days"
    if best_lag > 0
    else f"the S&P 500 leads RuneScape bonds by {abs(best_lag)} trading days"
    if best_lag < 0
    else "the strongest relationship is contemporaneous"
)
shuffle_pass = shuffle_pvalue < 0.05

display(
    Markdown(
        "## Section 3 Observations\n\n"
        f"The strongest return correlation appears at **lag {best_lag}** with Pearson correlation **{best_corr:.4f}** (p-value **{best_pvalue:.4g}**), meaning **{lag_direction_text}**. "
        f"Against the shuffle-test null, the observed best absolute correlation has an empirical p-value of **{shuffle_pvalue:.4f}**, so it **{'does' if shuffle_pass else 'does not'}** clearly beat what a broad lag search would often produce by chance. "
        "This lag becomes the candidate signal for the later multivariate, Granger, and out-of-sample checks, but it should only be trusted if those later sections continue to support it."
    )
)

## Section 4 — Multivariate Regression Against Known Indicators

A raw signal is not enough. The critical question is whether RuneScape bond returns still matter once more conventional explanatory variables are present. If the bond coefficient collapses after adding standard risk or sentiment indicators, then the apparent relationship is likely a proxy for broader market conditions rather than something genuinely new.

This section therefore adds two more established indicators. The VIX proxies for equity risk appetite and volatility demand, while the University of Michigan consumer sentiment index provides a slower-moving macro sentiment backdrop. If the RuneScape signal remains statistically meaningful after controlling for both, it becomes more plausible that the series captures something incremental rather than simply echoing well-known market state variables.

In [ ]:
analysis_start = analysis_df.index.min()
analysis_end = analysis_df.index.max()

umcsent_raw = load_or_fetch_fred_series("UMCSENT", start=analysis_start, end=analysis_end)
vix_raw = load_or_fetch_yfinance_series(VIX_SYMBOL, start=analysis_start, end=analysis_end)

umcsent_daily = umcsent_raw.rename(columns={"UMCSENT": "umcsent"}).reindex(analysis_df.index).ffill()
umcsent_daily["umcsent_change"] = umcsent_daily["umcsent"].pct_change()

vix_daily = vix_raw.rename(columns={f"{VIX_SYMBOL}_price": "vix_price"}).reindex(analysis_df.index).ffill()
vix_daily["vix_return"] = vix_daily["vix_price"].pct_change()

regression_df = analysis_df.join([umcsent_daily[["umcsent_change"]], vix_daily[["vix_return"]]], how="left")
regression_df["bond_signal"] = regression_df["bond_return"].shift(best_lag)
regression_df = regression_df.dropna(subset=["sp500_return", "bond_signal", "umcsent_change", "vix_return"]).copy()

X = regression_df[["bond_signal", "umcsent_change", "vix_return"]]
X = sm.add_constant(X)
y = regression_df["sp500_return"]
ols_model = sm.OLS(y, X).fit()

regression_results = pd.DataFrame(
    {
        "coefficient": ols_model.params,
        "pvalue": ols_model.pvalues,
        "tstat": ols_model.tvalues,
    }
)
display(regression_results)
print(f"R-squared: {ols_model.rsquared:.4f}")

rs_sig = bool(ols_model.pvalues.get("bond_signal", 1.0) < 0.05)
rs_coef = float(ols_model.params.get("bond_signal", np.nan))

display(
    Markdown(
        "## Section 4 Observations\n\n"
        f"After controlling for VIX returns and consumer-sentiment changes, the RuneScape bond signal carries a coefficient of **{rs_coef:.4f}** with p-value **{ols_model.pvalues.get('bond_signal', np.nan):.4g}**. "
        f"The full model explains **{ols_model.rsquared:.2%}** of daily S&P 500 return variation. "
        + (
            "The bond term remains statistically significant, which is consistent with the signal containing information that is not fully redundant with these comparison indicators."
            if rs_sig
            else "The bond term is not statistically significant once these known indicators are included, which suggests the raw relationship may mostly be a proxy for broader market-risk conditions rather than a novel signal."
        )
    )
)

## Section 5 — Granger Causality

Granger causality is not a claim about physical causation or economic mechanism. It is a narrower forecasting statement: does one series improve prediction of another series beyond the target series' own lagged history? That is useful here because a lead-lag correlation can still be spurious if the apparent leader does not add incremental predictive content once autoregressive structure is accounted for.

The reverse-direction test matters just as much as the forward one. If both directions look significant, the result is less supportive of a clean predictive story and more suggestive of common reaction to shared latent conditions or to broad market synchrony. The strongest version of the hypothesis would be significance from bond returns to the S&P 500 without a similarly strong reverse result.

In [ ]:
candidate_lags = sorted({lag for lag in [1, 5, 10, 30, abs(best_lag)] if lag > 0})


def run_granger_table(target: pd.Series, cause: pd.Series, lag_orders: Iterable[int], label: str) -> pd.DataFrame:
    records = []
    for lag_order in lag_orders:
        sample = pd.concat([target.rename("target"), cause.rename("cause")], axis=1).dropna()
        test_output = grangercausalitytests(sample[["target", "cause"]], maxlag=lag_order, verbose=False)
        fstat, pvalue, _, _ = test_output[lag_order][0]["ssr_ftest"]
        records.append({"direction": label, "lag_order": lag_order, "fstat": fstat, "pvalue": pvalue})
    return pd.DataFrame(records)


granger_forward = run_granger_table(analysis_df["sp500_return"], analysis_df["bond_return"], candidate_lags, "Bond -> S&P 500")
granger_reverse = run_granger_table(analysis_df["bond_return"], analysis_df["sp500_return"], candidate_lags, "S&P 500 -> Bond")
granger_results = pd.concat([granger_forward, granger_reverse], ignore_index=True)
display(granger_results)

forward_sig = granger_forward[granger_forward["pvalue"] < 0.05]
reverse_sig = granger_reverse[granger_reverse["pvalue"] < 0.05]

forward_text = ", ".join(str(x) for x in forward_sig["lag_order"].tolist()) or "none"
reverse_text = ", ".join(str(x) for x in reverse_sig["lag_order"].tolist()) or "none"

display(
    Markdown(
        "## Section 5 Observations\n\n"
        f"In the forward direction, RuneScape bond returns Granger-cause S&P 500 returns at lag orders **{forward_text}**. "
        f"In the reverse direction, S&P 500 returns Granger-cause RuneScape bond returns at lag orders **{reverse_text}**. "
        "If significance appears mainly in the forward direction, that supports the lead-lag story from Section 3. If the reverse direction is also strong, the safer interpretation is that both series are reacting to shared conditions rather than that one is uniquely informative about the other."
    )
)

## Section 6 — Rolling Correlation and Regime Stability

A weak average correlation can still be meaningful if it turns on and off in coherent regimes. In fact, regime dependence is often more informative than a single full-sample number because it points toward a mechanism that is only active under certain market conditions. A relationship that tightens during stress periods and disappears in calm periods implies something very different from a constant low-grade connection.

Rolling correlations make that time variation visible, while structural-break detection asks whether the observed changes are large enough to be treated as genuine regime shifts rather than visual noise. The notebook marks both data-defined breakpoints and known macro events so it is possible to ask whether the breaks align with recognizable external shocks or emerge in less obvious places.

In [ ]:
rolling_corr = pd.DataFrame(index=analysis_df.index)
rolling_corr["corr_60"] = analysis_df["bond_return"].rolling(60).corr(analysis_df["sp500_return"])
rolling_corr["corr_90"] = analysis_df["bond_return"].rolling(90).corr(analysis_df["sp500_return"])

break_series = rolling_corr["corr_90"].dropna()
break_algo = rpt.Binseg(model="l2").fit(break_series.to_numpy())
n_bkps = min(5, max(1, len(break_series) // 250))
break_idx = break_algo.predict(n_bkps=n_bkps)
break_dates = [break_series.index[i - 1] for i in break_idx[:-1] if i - 1 < len(break_series)]

fig, ax = plt.subplots(figsize=(15, 6))
rolling_corr.plot(ax=ax, linewidth=2)
for breakpoint in break_dates:
    ax.axvline(breakpoint, color="#c44e52", linestyle="--", alpha=0.75)
for event_date, label in [(pd.Timestamp("2020-03-01"), "COVID crash"), (pd.Timestamp("2022-03-16"), "Rate hikes start")]:
    ax.axvline(event_date, color="#dd8452", linestyle=":", alpha=0.9)
    ax.text(event_date, ax.get_ylim()[1] * 0.9, label, rotation=90, verticalalignment="top", color="#8c613c")
ax.set_title("Rolling bond / S&P 500 return correlation with structural breaks")
ax.set_ylabel("Rolling Pearson correlation")
ax.set_xlabel("Date")
plt.show()

break_text = ", ".join(date.strftime("%Y-%m-%d") for date in break_dates) or "none"

display(
    Markdown(
        "## Section 6 Observations\n\n"
        f"The rolling-correlation profile is not constant through time, and the structural-break routine flags breakpoints around **{break_text}**. "
        "The key interpretive question is whether those breaks cluster near known macro shocks such as the March 2020 COVID drawdown or the March 2022 start of the hiking cycle. Alignment with those dates would support a regime-dependent mechanism, while breaks far from obvious macro events would suggest the need for a more specific explanation or a closer look at model sensitivity."
    )
)

## Section 7 — Out of Sample Validation

The notebook now moves from explanation to forecast discipline. The date range is split into train, validation, and test segments so that decisions are not implicitly optimized on the final holdout period. The validation set is where the signal rule is chosen; the test set exists only to answer whether that locked rule still works on unseen data.

This separation is not bookkeeping. Touching the test set early turns it into another tuning set and makes the final accuracy number unreliable. If the signal only looks good after many informal tweaks that were influenced by the test period, then the notebook has not discovered a predictive relationship, it has discovered a way to overfit history.

In [ ]:
positive_lag_df = lag_corr_df[lag_corr_df["lag"] > 0].copy()
tradable_lag = int(
    best_lag
    if best_lag > 0
    else positive_lag_df.loc[positive_lag_df["correlation"].abs().idxmax(), "lag"]
)

prediction_df = analysis_df[["bond_return", "sp500_return"]].copy()
prediction_df["signal"] = prediction_df["bond_return"].shift(tradable_lag)
prediction_df = prediction_df.dropna().copy()

n = len(prediction_df)
train_end = int(n * 0.6)
valid_end = int(n * 0.8)

train_df = prediction_df.iloc[:train_end].copy()
valid_df = prediction_df.iloc[train_end:valid_end].copy()
test_df = prediction_df.iloc[valid_end:].copy()

train_corr = float(train_df[["signal", "sp500_return"]].corr().iloc[0, 1])
signal_orientation = 1.0 if np.isnan(train_corr) or train_corr >= 0 else -1.0
majority_direction = 1 if (train_df["sp500_return"] > 0).mean() >= 0.5 else -1
threshold_candidates = sorted(train_df["signal"].abs().quantile([0.0, 0.25, 0.5, 0.75]).unique())


def make_direction_predictions(
    signal: pd.Series,
    threshold: float,
    orientation: float,
    fallback_direction: int,
) -> pd.Series:
    raw = np.sign(signal * orientation)
    adjusted = raw.where(signal.abs() > threshold, fallback_direction)
    adjusted = adjusted.replace(0, fallback_direction)
    return adjusted.astype(int)


def evaluate_accuracy(
    frame: pd.DataFrame,
    threshold: float,
    orientation: float,
    fallback_direction: int,
) -> tuple[float, pd.DataFrame]:
    scored = frame.copy()
    scored["prediction"] = make_direction_predictions(scored["signal"], threshold, orientation, fallback_direction)
    scored["actual"] = np.where(scored["sp500_return"] >= 0, 1, -1)
    scored["correct"] = (scored["prediction"] == scored["actual"]).astype(int)
    return float(scored["correct"].mean()), scored


validation_scores = []
for threshold in threshold_candidates:
    accuracy, _ = evaluate_accuracy(valid_df, threshold, signal_orientation, majority_direction)
    validation_scores.append({"threshold": float(threshold), "validation_accuracy": accuracy})
validation_results = pd.DataFrame(validation_scores).sort_values(
    ["validation_accuracy", "threshold"],
    ascending=[False, True],
)
best_threshold = float(validation_results.iloc[0]["threshold"])

validation_accuracy, valid_scored = evaluate_accuracy(valid_df, best_threshold, signal_orientation, majority_direction)
test_accuracy, test_scored = evaluate_accuracy(test_df, best_threshold, signal_orientation, majority_direction)

walkforward_scored = pd.concat([valid_scored.assign(split="validation"), test_scored.assign(split="test")])
walkforward_scored["cumulative_accuracy"] = walkforward_scored["correct"].expanding().mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(
    walkforward_scored.index,
    walkforward_scored["cumulative_accuracy"],
    color=sns.color_palette("crest", 5)[3],
    linewidth=2,
)
ax.axvline(valid_scored.index.max(), color="#c44e52", linestyle="--", label="Validation/Test boundary")
ax.set_title("Cumulative directional accuracy across validation and test windows")
ax.set_ylabel("Cumulative accuracy")
ax.set_xlabel("Date")
ax.legend()
plt.show()

display(validation_results)

display(
    Markdown(
        "## Section 7 Observations\n\n"
        f"Using a tradable lead of **{tradable_lag} days**, the validation split selected an absolute-signal threshold of **{best_threshold:.4f}**. "
        f"Directional accuracy was **{validation_accuracy:.2%}** on validation and **{test_accuracy:.2%}** on the untouched test set. "
        + (
            "Test accuracy held up reasonably well, which is what the signal would need to do before any stronger predictive claim becomes credible."
            if test_accuracy >= validation_accuracy - 0.02
            else "Accuracy degraded materially from validation to test, which is a warning sign that the apparent signal may not be stable enough for real predictive use."
        )
    )
)

## Section 8 — Summary and Open Questions

The point of the notebook is not to accumulate loosely related charts. It is to force the idea through a sequence of filters and then summarize what survived. This final section gathers the key result from each stage so the overall case can be judged as a coherent argument rather than as isolated favorable fragments.

A credible conclusion should reconcile all of the evidence. If cross-correlation looks interesting but Granger causality, multivariate controls, and out-of-sample validation all fail, then the prudent conclusion is that the relationship is probably weak, unstable, or redundant. If several sections point in the same direction, the next question becomes mechanism: what feature of RuneScape bond trading could plausibly map onto broader market behavior, and under which regimes should that channel be strongest?

In [ ]:
stationarity_summary = "Returns used; price levels treated as non-stationary"
shuffle_summary = f"Empirical p-value {shuffle_pvalue:.4f}"
regression_summary = f"Bond coef p-value {ols_model.pvalues.get('bond_signal', np.nan):.4g}"
granger_summary = (
    f"Forward significant lags: {', '.join(map(str, forward_sig['lag_order'].tolist())) or 'none'}; "
    f"reverse: {', '.join(map(str, reverse_sig['lag_order'].tolist())) or 'none'}"
)
regime_summary = f"Breakpoints around {break_text}"
oos_summary = f"Validation {validation_accuracy:.2%}; test {test_accuracy:.2%}"

summary_table = pd.DataFrame(
    [
        {"section": "Stationarity", "key_result": stationarity_summary},
        {"section": "Best lag", "key_result": f"Lag {best_lag}, corr {best_corr:.4f}"},
        {"section": "Shuffle test", "key_result": shuffle_summary},
        {"section": "Regression controls", "key_result": regression_summary},
        {"section": "Granger", "key_result": granger_summary},
        {"section": "Regime stability", "key_result": regime_summary},
        {"section": "Out of sample", "key_result": oos_summary},
    ]
)

display(summary_table)

mechanism_text = (
    "Taken together, the evidence is most consistent with a relationship that is at least partly distinct from the comparison indicators and therefore worth deeper mechanism work."
    if rs_sig and shuffle_pass and not forward_sig.empty and test_accuracy > 0.5
    else "Taken together, the evidence is more consistent with a weak, unstable, or redundant relationship than with a strong standalone predictive signal."
)

display(
    Markdown(
        "## Synthesis\n\n"
        + mechanism_text
        + " The most plausible mechanism should now be judged by where the signal appeared strongest: if it concentrated in stress regimes and overlapped with VIX, a market-risk channel is more plausible; if it survived those controls and held out of sample, then the next step is to identify what unique behavioral or liquidity information RuneScape bonds might be capturing."
    )
)

display(
    Markdown(
        "## Open Questions and Next Steps\n\n"
        "1. What would strengthen the finding: repeat the same workflow on adjacent risk assets, alternative equity benchmarks, and additional in-game items to see whether the signal is specific to bonds or just a generic noisy risk proxy.\n"
        "2. What would falsify it: failure to survive longer out-of-sample windows, collapse after small specification changes, or equally strong reverse-direction results would all weaken the claim substantially.\n"
        "3. What additional data would help most: higher-frequency bond trading data, richer macro controls, and any direct measures of player activity or liquidity would make it easier to test a concrete mechanism rather than only a statistical association."
    )
)